Python datascript loader from local system+ the training script

In [43]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.10.0+cpu
Uninstalling torch-2.10.0+cpu:
  Successfully uninstalled torch-2.10.0+cpu
Found existing installation: torchvision 0.25.0+cpu
Uninstalling torchvision-0.25.0+cpu:
  Successfully uninstalled torchvision-0.25.0+cpu
Found existing installation: torchaudio 2.10.0+cpu
Uninstalling torchaudio-2.10.0+cpu:
  Successfully uninstalled torchaudio-2.10.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:0000:0100:01
     ━━━━

In [15]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import rasterio
from torch.utils.data import Dataset, DataLoader

In [16]:
import os
import torch
import rasterio
from torch.utils.data import Dataset
import re

class SEN12MSCR_Dataset(Dataset):
    def __init__(self, root_dir, split="train"):
        """
        split: "train" (0-2500 per season), "val" (2500-3000 per season), or "test" (3000-3500 per season)
        """
        self.triplets = []
        self.split = split
        seasons = ["ROIs1158_spring", "ROIs1868_summer", "ROIs1970_fall", "ROIs2017_winter"]

        for season in seasons:
            season_dir = os.path.join(root_dir, season)
            
            # Skip if the season directory doesn't exist to prevent crashes
            if not os.path.exists(season_dir):
                print(f"Warning: Directory not found - {season_dir}")
                continue
                
            s1_dir, s2c_dir, s2_dir = [os.path.join(season_dir, d) for d in ["s1", "s2_cloudy", "s2"]]

            # Create a mapping: ID -> filename
            def get_id(fname):
                match = re.search(r'(\d+_[p|P]\d+)', fname)
                return match.group(1) if match else None

            # Map IDs to filenames, ignoring hidden files
            s1_map = {get_id(f): f for f in os.listdir(s1_dir) if f.endswith('.tif')}
            s2c_map = {get_id(f): f for f in os.listdir(s2c_dir) if f.endswith('.tif')}
            s2_map = {get_id(f): f for f in os.listdir(s2_dir) if f.endswith('.tif')}

            # Find common IDs across all three modalities
            common_ids = set(s1_map.keys()) & set(s2c_map.keys()) & set(s2_map.keys())
            if None in common_ids:
                common_ids.remove(None)
            
            # MUST sort the IDs so the splits are deterministic across runs
            sorted_ids = sorted(list(common_ids))
            
            # --- THE SPLIT LOGIC ---
            if self.split == "train":
                split_ids = sorted_ids[:2500]
            elif self.split == "val":
                split_ids = sorted_ids[2500:3000]
            elif self.split == "test":
                split_ids = sorted_ids[3000:3500]
            else:
                split_ids = sorted_ids # Fallback uses all data

            # Append the paths for this specific split
            for uid in split_ids:
                self.triplets.append({
                    "s1": os.path.join(s1_dir, s1_map[uid]),
                    "s2c": os.path.join(s2c_dir, s2c_map[uid]),
                    "s2_clear": os.path.join(s2_dir, s2_map[uid])
                })
                
        print(f"Initialized '{self.split}' dataset with {len(self.triplets)} total images.")

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        paths = self.triplets[idx]
        
        with rasterio.open(paths["s1"]) as src:
            s1 = torch.tensor(src.read().astype("float32"))
            
        with rasterio.open(paths["s2c"]) as src:
            s2c = torch.tensor(src.read().astype("float32"))
            
        with rasterio.open(paths["s2_clear"]) as src:
            s2_clear = torch.tensor(src.read().astype("float32"))
            
        # --- CRITICAL FIX: DIFFUSION NORMALIZATION ---
        # 1. Scale Sentinel-2 reflectances from [0, 10000] down to [0.0, 1.0]
        s2c = torch.clamp(s2c / 10000.0, 0.0, 1.0)
        s2_clear = torch.clamp(s2_clear / 10000.0, 0.0, 1.0)
        
        # 2. Shift to [-1.0, 1.0] to match the True Noise distribution N(0,1)
        s2c = (s2c * 2.0) - 1.0
        s2_clear = (s2_clear * 2.0) - 1.0
            
        return s1, s2c, s2_clear

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LayerNorm2d(nn.Module):
    """
    Standard LayerNorm expects the channel dimension last. 
    This wrapper cleanly applies it to (B, C, H, W) image tensors.
    """
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
        
    def forward(self, x):
        return self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

class SSA(nn.Module):
    """
    Split Spatial Attention
    Splits the input tensor into two halves along the channel dimension
    and performs an element-wise product.
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Split channels into two equal halves
        c = x.size(1)
        x1, x2 = torch.split(x, c // 2, dim=1)
        # Element-wise product acts as a gating mechanism
        return x1 * x2

class SCA(nn.Module):
    """
    Spatial Channel Attention
    Applies Global Max Pooling to one half and Global Average Pooling to the other,
    uses Conv1x1 to generate attention weights, and recalibrates the branches.
    """
    def __init__(self, split_channels):
        super().__init__()
        # Conv1x1 layers to generate attention vectors from the pooled outputs
        self.conv_gmp = nn.Conv2d(split_channels, split_channels, kernel_size=1)
        self.conv_gap = nn.Conv2d(split_channels, split_channels, kernel_size=1)

    def forward(self, x):
        c = x.size(1)
        x1, x2 = torch.split(x, c // 2, dim=1)

        # Branch 1: Global Max Pooling (GMP)
        gmp = F.adaptive_max_pool2d(x1, (1, 1))
        attn_gmp = self.conv_gmp(gmp)
        out1 = x1 * attn_gmp # Recalibrate original left split

        # Branch 2: Global Average Pooling (GAP)
        gap = F.adaptive_avg_pool2d(x2, (1, 1))
        attn_gap = self.conv_gap(gap)
        out2 = x2 * attn_gap # Recalibrate original right split

        # Channel-wise concatenation
        return torch.cat([out1, out2], dim=1)

class SpatialExtractionModule(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln = LayerNorm2d(dim)
        # Expand channels x2 so SSA can halve them back to dim
        self.conv_expand = nn.Conv2d(dim, dim * 2, kernel_size=1)
        # Depth-wise convolution (groups = in_channels)
        self.dconv = nn.Conv2d(dim * 2, dim * 2, kernel_size=3, padding=1, groups=dim * 2)
        
        self.ssa = SSA() 
        # SCA takes the dim, splits it internally, and concatenates back to dim
        self.sca = SCA(split_channels=dim // 2)
        
        self.conv_proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        res = x
        x = self.ln(x)
        x = self.conv_expand(x)
        x = self.dconv(x)
        x = self.ssa(x)
        x = self.sca(x)
        x = self.conv_proj(x)
        return x + res # Element-wise summation

class FeatureRecalibrationModule(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln = LayerNorm2d(dim)
        # Expand for SSA
        self.conv_expand = nn.Conv2d(dim, dim * 2, kernel_size=1)
        self.ssa = SSA()
        self.conv_proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, x):
        res = x
        x = self.ln(x)
        x = self.conv_expand(x)
        x = self.ssa(x)
        x = self.conv_proj(x)
        return x + res

class ConditionEncoderBlock(nn.Module):
    def __init__(self, in_channels, dim, downsample=True):
        super().__init__()
        # Initial projection if input channels don't match the internal dimension
        self.proj = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1) if in_channels != dim else nn.Identity()
        
        self.spatial_extraction = SpatialExtractionModule(dim)
        self.feature_recalibration = FeatureRecalibrationModule(dim)
        
        # BUG FIX: Keep channels at 'dim'. Do not use 'dim * 2' here!
        # The next block's 'proj' layer will handle the channel doubling.
        if downsample:
            self.downsample = nn.Conv2d(dim, dim, kernel_size=3, stride=2, padding=1)
        else:
            self.downsample = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = self.spatial_extraction(x)
        x = self.feature_recalibration(x)
        return self.downsample(x)

# ==========================================
# DIAGNOSTIC TEST BLOCK
# ==========================================
if __name__ == "__main__":
    # Simulating a batch of 4 cloudy Sentinel-2 images (13 channels, 256x256)
    cloudy_input = torch.randn(4, 13, 256, 256)
    
    # Initialize the encoder (mapping 13 input channels to an internal dim of 64)
    encoder = ConditionEncoderBlock(in_channels=13, dim=64, downsample=True)
    
    output = encoder(cloudy_input)
    
    print(f"Input Shape:  {cloudy_input.shape}")
    print(f"Output Shape: {output.shape}")

import torch
import torch.nn as nn
import math

class SinusoidalPositionEmbeddings(nn.Module):
    """
    The 'sin(t)' portion of the equation.
    Converts a single scalar time step into a multi-frequency vector.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        
        # Calculate frequencies
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        
        # Apply frequencies to the time steps
        embeddings = time[:, None] * embeddings[None, :]
        
        # Interleave sine and cosine (Standard DDPM implementation)
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class TimeEncoder(nn.Module):
    """
    The full Ft = Reshape(MLP(sin(t))) implementation.
    """
    def __init__(self, time_dim):
        super().__init__()
        self.time_dim = time_dim
        
        # The base dimension is usually smaller, then expanded by the MLP
        base_dim = time_dim // 4
        self.sinusoidal_embed = SinusoidalPositionEmbeddings(base_dim)
        
        # The 'MLP' portion of the equation
        # Note: SiLU (Swish) is the standard activation function for diffusion models
        self.mlp = nn.Sequential(
            nn.Linear(base_dim, time_dim),
            nn.SiLU(), 
            nn.Linear(time_dim, time_dim)
        )

    def forward(self, t):
        # 1. Get sinusoidal embedding: sin(t)
        x = self.sinusoidal_embed(t)
        
        # 2. Pass through the Multi-Layer Perceptron: MLP(...)
        x = self.mlp(x)
        
        # 3. The 'Reshape(...)' portion of the equation
        # We reshape from (Batch, Channels) to (Batch, Channels, 1, 1) 
        # so it can be mathematically added to the (Batch, Channels, Height, Width) image tensors later.
        F_t = x.view(-1, self.time_dim, 1, 1)
        
        return F_t

# ==========================================
# DIAGNOSTIC TEST BLOCK
# ==========================================
if __name__ == "__main__":
    # Simulating a batch of 4 different time steps (e.g., t=50, t=100, t=500, t=999)
    # Time steps are usually integers in DDPMs
    t_input = torch.tensor([50, 100, 500, 999], dtype=torch.float32)
    
    # Initialize the encoder (Targeting a 256-channel injection dimension)
    time_encoder = TimeEncoder(time_dim=256)
    
    # Forward pass
    F_t = time_encoder(t_input)
    
    print(f"Input Time Steps: {t_input.shape} -> {t_input.tolist()}")
    print(f"Output F_t Shape: {F_t.shape}")



import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """Standard U-Net Double Convolution Block"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class SAR_Prior_UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=13, hidden_dim=64):
        super().__init__()
        
        # --- Encoder (Downsampling) ---
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        # Bottleneck
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        
        # --- Decoder (Upsampling) ---
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, hidden_dim) 
        
        # --- CRITICAL FIXES FOR STAGE 2 ALIGNMENT ---
        # Projects f_sar out of the strict ReLU bounds so it can fuse mathematically
        self.f_sar_proj = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=1)
        
        self.outc = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1),
            # Tanh bounds the output to [-1, 1] to match the newly normalized dataset
            nn.Tanh() 
        )

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)
        
        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)
        
        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        
        # Capture Penultimate Hidden Representation
        x_hidden = self.conv4(x) 
        
        # Project it for Cross-Modal Fusion
        f_sar = self.f_sar_proj(x_hidden)
        
        # Generate the Simulated Optical Image
        simulated_img = self.outc(f_sar)
        
        return f_sar, simulated_img



import torch
import torch.nn as nn

class CrossModalGatedFusion(nn.Module):
    """
    Dynamically blends SAR and Optical features. 
    Learns to trust SAR in clouded regions and Optical in clear regions.
    """
    def __init__(self, dim):
        super().__init__()
        
        # The 'Gate' looks at both features and calculates a spatial probability map.
        # It outputs a single channel (1 = Trust Optical, 0 = Trust SAR)
        self.spatial_gate = nn.Sequential(
            nn.Conv2d(dim * 2, dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True),
            # Compress down to a single 2D attention mask
            nn.Conv2d(dim, 1, kernel_size=3, padding=1),
            nn.Sigmoid() # Bounds the mask strictly between 0.0 and 1.0
        )
        
        # Optional: A final smoothing layer after the blend
        self.smooth = nn.Conv2d(dim, dim, kernel_size=3, padding=1)

    def forward(self, f_sar, f_c):
        # 1. Concatenate them solely to calculate the mask
        combined = torch.cat([f_sar, f_c], dim=1)
        
        # 2. Generate the spatial alpha mask (Shape: Batch, 1, Height, Width)
        # This is essentially the network teaching itself where the clouds are!
        trust_optical_mask = self.spatial_gate(combined)
        trust_sar_mask = 1.0 - trust_optical_mask
        
        # 3. The Smart Blend (Adaptive Muxing)
        # Where optical is clouded (mask ~ 0), SAR is amplified (1 - 0 = 1)
        fused_features = (f_c * trust_optical_mask) + (f_sar * trust_sar_mask)
        
        return self.smooth(fused_features)

# ==========================================
# DIAGNOSTIC TEST BLOCK
# ==========================================
if __name__ == "__main__":
    # Simulating the hidden representations (e.g., 64 channels)
    f_sar = torch.randn(4, 64, 128, 128)
    f_c = torch.randn(4, 64, 128, 128)
    
    fusion_block = CrossModalGatedFusion(dim=64)
    
    fused_output = fusion_block(f_sar, f_c)
    
    print(f"SAR Feature Shape:     {f_sar.shape}")
    print(f"Optical Feature Shape: {f_c.shape}")
    print(f"Smart Fused Output:    {fused_output.shape}")



import torch
import torch.nn as nn

class FullTCFBlock(nn.Module):
    """
    The complete Time and Condition Fusion Block.
    Injects F_t and F_c into the residual stream.
    """
    def __init__(self, dim, time_dim, cond_dim):
        super().__init__()
        
        # We assume SpatialExtractionModule and FeatureRecalibrationModule 
        # are defined exactly as we built them in the Condition Encoder step.
        self.spatial_extraction = SpatialExtractionModule(dim)
        self.feature_recalibration = FeatureRecalibrationModule(dim)
        
        # Projection layers to ensure the injected features match the block's channel dimension
        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, dim)
        )
        # If the multiscale condition already matches 'dim', just pass it through.
        self.cond_proj = nn.Conv2d(cond_dim, dim, kernel_size=1) if cond_dim != dim else nn.Identity()

    def forward(self, x, f_t, f_c):
        # 1. Time Injection (F_t)
        # Project time embedding and reshape to (Batch, dim, 1, 1) for broadcasting
        t_emb = self.time_proj(f_t.view(f_t.size(0), -1))
        t_emb = t_emb.unsqueeze(-1).unsqueeze(-1) 
        
        x = x + t_emb # Element-wise Summation
        x = self.spatial_extraction(x)
        
        # 2. Condition Injection (F_c)
        c_emb = self.cond_proj(f_c)
        
        x = x + c_emb # Element-wise Summation
        x = self.feature_recalibration(x)
        
        return x

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, cond_dim):
        super().__init__()
        self.tcf = FullTCFBlock(dim=out_channels, time_dim=time_dim, cond_dim=cond_dim)
        self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x, f_t, f_c):
        x = self.downsample(x) # F_{e+1} = DownSample(...)
        x = self.tcf(x, f_t, f_c)
        return x

class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, cond_dim):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        # in_channels is the decoder input. skip_x is half that.
        # So in_channels // 2 (upsampled) + skip_x (original) = in_channels
        self.conv_reduce = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.tcf = FullTCFBlock(dim=out_channels, time_dim=time_dim, cond_dim=cond_dim)

    def forward(self, x, skip_x, f_t, f_c):
        x = self.upsample(x)
        x = torch.cat([x, skip_x], dim=1) 
        x = self.conv_reduce(x) # Project back to out_channels
        x = self.tcf(x, f_t, f_c)
        return x

class DenoisingAutoencoder(nn.Module):
    def __init__(self, in_channels=13, base_dim=64, time_dim=256):
        super().__init__()
        
        # Initial projection of the noisy image (y_t)
        self.init_conv = nn.Conv2d(in_channels, base_dim, kernel_size=3, padding=1)
        
        # ENCODER
        self.down1 = DownBlock(base_dim, base_dim * 2, time_dim, cond_dim=base_dim * 2)
        self.down2 = DownBlock(base_dim * 2, base_dim * 4, time_dim, cond_dim=base_dim * 4)
        self.down3 = DownBlock(base_dim * 4, base_dim * 8, time_dim, cond_dim=base_dim * 8)
        
        # BOTTLENECK 
        self.mid_tcf1 = FullTCFBlock(base_dim * 8, time_dim, cond_dim=base_dim * 8)
        self.mid_tcf2 = FullTCFBlock(base_dim * 8, time_dim, cond_dim=base_dim * 8)
        
        # DECODER
        self.up1 = UpBlock(base_dim * 8, base_dim * 4, time_dim, cond_dim=base_dim * 4)
        self.up2 = UpBlock(base_dim * 4, base_dim * 2, time_dim, cond_dim=base_dim * 2)
        
        # up3 receives an upsampled version of f_c_list[0]. 
        # f_c_list[0] has 'base_dim * 2' channels, so cond_dim must reflect that!
        self.up3 = UpBlock(base_dim * 2, base_dim, time_dim, cond_dim=base_dim * 2)
        
        # Final output head predicting the noise
        self.out_conv = nn.Conv2d(base_dim, in_channels, kernel_size=3, padding=1)

    def forward(self, y_t, f_t, f_c_list):
        x0 = self.init_conv(y_t)
        
        # Encoding Path
        x1 = self.down1(x0, f_t, f_c_list[0])  # x1: 128x128 spatial
        x2 = self.down2(x1, f_t, f_c_list[1])  # x2: 64x64 spatial
        x3 = self.down3(x2, f_t, f_c_list[2])  # x3: 32x32 spatial
        
        # Bottleneck Path
        x_mid = self.mid_tcf1(x3, f_t, f_c_list[3]) # 32x32 spatial
        x_mid = self.mid_tcf2(x_mid, f_t, f_c_list[3])
        
        # Decoding Path
        # Align spatial dimensions: up1 outputs 64x64 -> pass f_c_list[1]
        x = self.up1(x_mid, x2, f_t, f_c_list[1])
        
        # Align spatial dimensions: up2 outputs 128x128 -> pass f_c_list[0]
        x = self.up2(x, x1, f_t, f_c_list[0])
        
        # up3 outputs 256x256. We interpolate f_c_list[0] to match the spatial size.
        f_c_upsampled = F.interpolate(f_c_list[0], size=(256, 256), mode='bilinear', align_corners=False)
        x = self.up3(x, x0, f_t, f_c_upsampled)
        
        # Predict the noise!
        predicted_noise = self.out_conv(x)
        return predicted_noise




Input Shape:  torch.Size([4, 13, 256, 256])
Output Shape: torch.Size([4, 64, 128, 128])
Input Time Steps: torch.Size([4]) -> [50.0, 100.0, 500.0, 999.0]
Output F_t Shape: torch.Size([4, 256, 1, 1])
SAR Feature Shape:     torch.Size([4, 64, 128, 128])
Optical Feature Shape: torch.Size([4, 64, 128, 128])
Smart Fused Output:    torch.Size([4, 64, 128, 128])


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiScaleConditionEncoder(nn.Module):
    """
    Processes the cloudy optical image and extracts a list of 4 feature maps 
    at decreasing spatial resolutions to inject into the Denoising U-Net.
    """
    # Ensure default in_channels matches the output of your CrossModalGatedFusion (64)
    def __init__(self, in_channels=64, base_dim=64):
        super().__init__()
        
        # Level 0: 64 -> 128 (Spatial: 256x256 -> 128x128)
        self.level0 = ConditionEncoderBlock(in_channels, base_dim * 2, downsample=True)
        
        # Level 1: 128 -> 256 (Spatial: 128x128 -> 64x64)
        self.level1 = ConditionEncoderBlock(base_dim * 2, base_dim * 4, downsample=True)
        
        # Level 2: 256 -> 512 (Spatial: 64x64 -> 32x32)
        self.level2 = ConditionEncoderBlock(base_dim * 4, base_dim * 8, downsample=True)
        
        # Level 3: 512 -> 512 (Bottleneck, no downsample. Spatial: 32x32 -> 32x32)
        self.level3 = ConditionEncoderBlock(base_dim * 8, base_dim * 8, downsample=False)

    def forward(self, cloudy_img):
        f_c_0 = self.level0(cloudy_img)
        f_c_1 = self.level1(f_c_0)
        f_c_2 = self.level2(f_c_1)
        f_c_3 = self.level3(f_c_2)
        
        # Returns [highest_res, mid_res, low_res, lowest_res]
        return [f_c_0, f_c_1, f_c_2, f_c_3]

In [19]:
class CosineNoiseSchedule:
    def __init__(self, num_timesteps=1000, s=0.008, device="cuda"):
        self.num_timesteps = num_timesteps
        self.device = device
        
        # Generate the cosine alpha_bar sequence
        steps = num_timesteps + 1
        x = torch.linspace(0, num_timesteps, steps, dtype=torch.float32)
        alphas_cumprod = torch.cos(((x / num_timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        
        # Derive betas
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        self.betas = torch.clip(betas, 0.0001, 0.999).to(device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

    def add_noise(self, original_images, noise, timesteps):
        batch_size = original_images.size(0)
        sqrt_alpha_t = self.sqrt_alphas_cumprod[timesteps].view(batch_size, 1, 1, 1)
        sqrt_one_minus_alpha_t = self.sqrt_one_minus_alphas_cumprod[timesteps].view(batch_size, 1, 1, 1)
        
        return sqrt_alpha_t * original_images + sqrt_one_minus_alpha_t * noise

In [20]:
import torchvision.models as models

class VGGPerceptualLoss(nn.Module):
    def __init__(self, device="cuda"):
        super().__init__()
        # Load pre-trained VGG16
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features.eval().to(device)
        
        # Extract features up to the relu2_2 layer (index 8)
        self.feature_extractor = nn.Sequential(*list(vgg.children())[:9])
        
        # Freeze VGG parameters
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
            
        # ImageNet Normalization values
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device))
        self.mse = nn.MSELoss()

    def forward(self, generated_img, target_img):
        # Extract RGB bands from Sentinel-2 (Assuming B4, B3, B2 are indices 3, 2, 1)
        # Adjust these indices if your dataset's band order is different
        gen_rgb = generated_img[:, 1:4, :, :] 
        tgt_rgb = target_img[:, 1:4, :, :]
        
        # Normalize to ImageNet statistics
        gen_rgb = (gen_rgb - self.mean) / self.std
        tgt_rgb = (tgt_rgb - self.mean) / self.std
        
        # Extract VGG features and compute MSE between them
        gen_features = self.feature_extractor(gen_rgb)
        tgt_features = self.feature_extractor(tgt_rgb)
        
        return self.mse(gen_features, tgt_features)

In [21]:
Pre training of the SAR Unet

SyntaxError: invalid syntax (329195316.py, line 1)

In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

def pretrain_sar_unet(train_loader, val_loader=None, epochs=30, device="cuda"):
    print("--- Starting Stage 1: SAR U-Net Pre-training ---")
    
    # 1. Initialize ONLY the SAR Model
    sar_unet = SAR_Prior_UNet(in_channels=2, out_channels=13, hidden_dim=64).to(device)
    
    # 2. Optimizer specifically for SAR
    optimizer = optim.AdamW(sar_unet.parameters(), lr=2e-4, weight_decay=1e-4)
    
    # 3. Loss Functions (Geometric + Visual)
    l1_loss = nn.L1Loss()
    perceptual_loss = VGGPerceptualLoss(device=device) # Uses the class we defined earlier
    
    # Track the best loss to save the best model
    best_loss = float('inf')
    save_path = "/kaggle/working/sar_unet_best.pth"

    # 4. Training Loop
    for epoch in range(epochs):
        sar_unet.train()
        epoch_loss = 0.0
        
        for batch_idx, (sar_img, cloudy_img, clear_img) in enumerate(train_loader):
            # We don't need cloudy_img for Stage 1!
            sar_img = sar_img.to(device)
            clear_img = clear_img.to(device)

            optimizer.zero_grad()

            # Forward Pass: Generate optical simulation from SAR
            f_sar, simulated_optical = sar_unet(sar_img)
            
            # Calculate Losses
            loss_l1 = l1_loss(simulated_optical, clear_img)
            loss_perc = perceptual_loss(simulated_optical, clear_img)
            
            # Total SAR Loss (You can adjust the 0.1 weight if needed)
            total_loss = loss_l1 + (0.1 * loss_perc)
            
            # Backpropagation
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch} | Batch {batch_idx} | L1 Loss: {loss_l1.item():.4f} | Perc Loss: {loss_perc.item():.4f}")

        # Average loss for the epoch
        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"*** Epoch {epoch} Complete | Avg Loss: {avg_epoch_loss:.4f} ***")
        
        # Save the model if it improved
        if avg_epoch_loss < best_loss:
            best_loss = avg_epoch_loss
            torch.save(sar_unet.state_dict(), save_path)
            print(f" -> Best model saved to {save_path}")

    print("--- Stage 1 Complete ---")
    return save_path

In [23]:
@torch.no_grad()
def generate_diffusion_samples(model_dict, sar_img, cloudy_img, noise_scheduler, device="cuda"):
    """
    Iteratively denoises a random tensor to generate the final clear image prediction.
    model_dict contains the frozen and active network components.
    """
    sar_unet = model_dict['sar_unet']
    cloudy_proj = model_dict['cloudy_proj']
    fusion_module = model_dict['fusion_module']
    condition_encoder = model_dict['condition_encoder']
    time_encoder = model_dict['time_encoder']
    diffusion_unet = model_dict['diffusion_unet']
    
    # 1. Get Conditions
    f_sar, _ = sar_unet(sar_img)
    f_c = cloudy_proj(cloudy_img)
    fused_condition = fusion_module(f_sar, f_c)
    f_c_list = condition_encoder(fused_condition)
    
    # 2. Start from pure Gaussian noise
    b, c, h, w = cloudy_img.shape
    x_t = torch.randn((b, c, h, w), device=device)
    
    # 3. Iterative Denoising (Reverse Process)
    # Loop backwards from T-1 to 0
    for i in reversed(range(0, noise_scheduler.num_timesteps)):
        t_tensor = torch.full((b,), i, device=device, dtype=torch.long)
        f_t = time_encoder(t_tensor.float())
        
        # Predict the noise
        predicted_noise = diffusion_unet(x_t, f_t, f_c_list)
        
        # Diffusion Math: Subtract the predicted noise scaled by schedule parameters
        alpha_t = noise_scheduler.alphas[i]
        alpha_bar_t = noise_scheduler.alphas_cumprod[i]
        beta_t = noise_scheduler.betas[i]
        
        # x_{t-1} = (1 / sqrt(alpha_t)) * (x_t - ((1 - alpha_t) / sqrt(1 - alpha_bar_t)) * predicted_noise)
        x_t = (1.0 / torch.sqrt(alpha_t)) * (x_t - ((1.0 - alpha_t) / torch.sqrt(1.0 - alpha_bar_t)) * predicted_noise)
        
        # Add Langevin noise if not the final step
        if i > 0:
            z = torch.randn_like(x_t)
            x_t = x_t + torch.sqrt(beta_t) * z
            
    # The final x_0 is our generated image
    return torch.clamp(x_t, -1.0, 1.0) # Clamp to valid image rang

In [ ]:
Training the actual diffusion using the frozen SAR Unet

In [39]:
def train_diffusion_model_stage2(train_loader, val_loader, epochs=50, device="cuda", resume_path=None):
    # ... [Keep your initialization code here] ...
    
    # --- FIXED RESUME LOGIC ---
    start_epoch = 0 # Define it here so it always exists
    if resume_path and os.path.exists(resume_path):
        print(f"Loading checkpoint from {resume_path}...")
        checkpoint = torch.load(resume_path, map_location=device)
        
        cloudy_proj.load_state_dict(checkpoint['cloudy_proj'])
        fusion_module.load_state_dict(checkpoint['fusion_module'])
        condition_encoder.load_state_dict(checkpoint['condition_encoder'])
        time_encoder.load_state_dict(checkpoint['time_encoder'])
        diffusion_unet.load_state_dict(checkpoint['diffusion_unet'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        
        start_epoch = checkpoint['epoch'] + 1
        print(f"Successfully resumed! Starting from epoch {start_epoch}")
    else:
        print("No checkpoint found. Starting from scratch.")

    print("--- Starting Stage 2 Training ---")
    # ... [The rest of your try/except loop follows] ...
    # ... [Keep your initialization code here] ...
    
    try:
        for epoch in range(start_epoch, epochs):
            # ... [Training loop remains the same] ...
            # Ensure you save a 'last_batch' sample for visualization
            for batch_idx, (sar_img, cloudy_img, clear_img) in enumerate(train_loader):
                # ... (your training steps) ...
                
                # Capture the last batch for visualization later
                last_sar, last_cloudy, last_clear = sar_img, cloudy_img, clear_img
            
            # Validation & Visualization
            if (epoch + 1) % 5 == 0:
                print(f"\n--- Running Validation & Generation for Epoch {epoch} ---")
                # Set all models to eval mode
                models = [cloudy_proj, fusion_module, condition_encoder, diffusion_unet, time_encoder]
                for m in models: m.eval()
                
                val_mse, psnr_total, ssim_total = 0.0, 0.0, 0.0
                
                with torch.no_grad():
                    # 1. Training Visualization using the captured last_batch
                    train_generated = generate_diffusion_samples(model_dict, last_sar.to(device), last_cloudy.to(device), noise_scheduler, device)
                    visualize_results(last_sar.to(device), last_cloudy.to(device), train_generated, last_clear.to(device), epoch, title_prefix="Train")

                    # 2. Validation Loop
                    for val_idx, (v_sar, v_cloudy, v_clear) in enumerate(val_loader):
                        v_sar, v_cloudy, v_clear = v_sar.to(device), v_cloudy.to(device), v_clear.to(device)
                        
                        # Calculate MSE
                        f_sar, _ = sar_unet(v_sar)
                        t = torch.randint(0, 1000, (v_clear.size(0),), device=device).long()
                        noisy_image = noise_scheduler.add_noise(v_clear, torch.randn_like(v_clear), t)
                        
                        f_c = cloudy_proj(v_cloudy)
                        f_c_list = condition_encoder(fusion_module(f_sar, f_c))
                        pred_noise = diffusion_unet(noisy_image, time_encoder(t.float()), f_c_list) 
                        val_mse += mse_loss(pred_noise, torch.randn_like(v_clear)).item()
                        
                        # Generate & Metrics (First 5 batches only)
                        if val_idx < 5: 
                            v_gen = generate_diffusion_samples(model_dict, v_sar, v_cloudy, noise_scheduler, device)
                            psnr_total += psnr_metric(v_gen, v_clear).item()
                            ssim_total += ssim_metric(v_gen, v_clear).item()
                            if val_idx == 0:
                                visualize_results(v_sar, v_cloudy, v_gen, v_clear, epoch, title_prefix="Validation")

                print(f"*** Val Stats | MSE: {val_mse/len(val_loader):.4f} | PSNR: {psnr_total/5.0:.2f} dB | SSIM: {ssim_total/5.0:.4f} ***\n")

            # Checkpoint Save
            # ... (your checkpoint dictionary code) ...
    except Exception as e:
        print(f"\n[!] Training error: {e}")
        raise e
    return "/kaggle/working/diffusion_latest_ckpt.pth"

In [44]:
# 1. Setup the path to your Kaggle dataset
DATA_DIR = "/kaggle/input/datasets/raghavk2311/sen12mscr-10k-triplets/extracted_data"

# 2. Initialize the datasets
train_dataset = SEN12MSCR_Dataset(root_dir=DATA_DIR, split="train")
val_dataset   = SEN12MSCR_Dataset(root_dir=DATA_DIR, split="val")

# 3. Create your loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

# =====================================================================
# THE TWO-STAGE EXECUTION PIPELINE
# =====================================================================
# Pass the resume path. If the file doesn't exist yet, it will just start from epoch 0.
ckpt_path = "/kaggle/working/diffusion_latest_ckpt.pth"
# STAGE 1: Pre-train the SAR feature extractor (runs for ~30 epochs)
# This will save a file called "sar_unet_best.pth" in Kaggle's working directory.
saved_model_path = pretrain_sar_unet(train_loader, epochs=30, device="cuda")

train_diffusion_model_stage2(
    train_loader,
    val_loader,
    epochs=50,
    device="cuda",
    resume_path=ckpt_path
)


# STAGE 2: Train the Diffusion model using the frozen SAR features
# Ensure your train_diffusion_model_stage2 function has the line uncommented that says:
# sar_unet.load_state_dict(torch.load("/kaggle/working/sar_unet_best.pth"))


Initialized 'train' dataset with 9544 total images.
Initialized 'val' dataset with 913 total images.
--- Starting Stage 1: SAR U-Net Pre-training ---


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
How to Fix the Dead Gradients
Since your architecture code is structurally sound, a model getting stuck at the 1.0 baseline is almost always caused by a data scaling issue. Check these three things immediately:

1. Input Image Normalization (The Most Likely Culprit)
Diffusion models strictly require your input images (clear_img) to be normalized to the range [-1.0, 1.0].

If your Sentinel-2 images are currently in the [0, 255] range, or even the [0, 1] range, the forward diffusion mathematics will break.

Adding N(0,1) noise to an image scaled [0, 255] makes the noise invisible to the network. Adding it to [0, 1] heavily skews the signal-to-noise ratio.

The Fix: In your DataLoader, ensure you apply transforms.Normalize(mean=[0.5, ...], std=[0.5, ...]) so the max pixel value is 1.0 and the min is -1.0.

2. Verify the Network Output Bounds
Double-check that the very last layer of your DenoisingAutoencoder (self.out_conv) does not have a Sigmoid or ReLU activation function applied to it anywhere in your forward pass.

The model must predict true_noise, which contains negative numbers (often down to -3.0). If you accidentally clamp the output to positive numbers with a ReLU, the loss will permanently flatline around 1.0.

3. SAR Prior Output Scaling
Your frozen SAR U-Net currently outputs a simulated optical image using a Sigmoid activation (range [0, 1]). If your clear_img dataset is correctly scaled to [-1, 1] for the diffusion model, but the SAR prior is passing [0, 1] representations into the fusion block, the conflicting feature scales can cause the optimizer to stall.